In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# **Spark UI**

In [2]:
spark = SparkSession.builder.appName(
    "Counting word occurences from a book, one more time."
).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/15 13:08:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.sparkContext

<SparkContext master=local[*] appName=Counting word occurences from a book, one more time.>

In [4]:
results = (
    spark.read.text("../data/gutenberg_books/*.txt")
        .select(F.split(F.col("value"), " ").alias("line"))
        .select(F.explode(F.col("line")).alias("word"))
        .select(F.lower(F.col("word")).alias("word"))
        .select(F.regexp_extract(F.col("word"), "[a-z']+", 0).alias("word"))
        .where(F.col("word") != "")
        .groupby(F.col("word"))
        .count()
)

26/11/15 13:09:01 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/gutenberg_books/*.txt.
java.io.FileNotFoundException: File ../data/gutenberg_books/*.txt does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catal

In [5]:
results.orderBy(F.col("count").desc()).show(10)

[Stage 0:>                                                          (0 + 3) / 3]

+----+-----+
|word|count|
+----+-----+
| the|39188|
| and|24292|
|  of|21234|
|  to|20581|
|   i|15151|
|   a|14564|
|  in|12857|
|that| 9900|
|  it| 9451|
| was| 8939|
+----+-----+
only showing top 10 rows


![](../images/resources_Spark.png)

Apache Spark’s resource management model is designed around a **Master-Worker architecture** managed by a cluster manager (such as Apache YARN, Kubernetes, or standalone Spark Master). Spark decouples physical cluster resource allocation from logical task execution, using a finely tuned **Unified Memory Manager** and a core-based task concurrency engine.

---

## **🏛️ 1. High-Level Architecture: Driver & Executors**

When you submit a PySpark application, Spark spins up two main types of processes:

* **The Driver:** The central coordinator. It runs your main Python script, builds the Directed Acyclic Graph (DAG), analyzes and optimizes queries via the Catalyst Optimizer, breaks execution plans into stages and tasks, and schedules tasks across executors.
* **The Executors:** The worker processes running on cluster nodes. Executors are responsible for actually executing tasks assigned by the driver, reading/writing data from storage, and holding cached data in memory or disk.

---

## **⚙️ 2. CPU and Task Management**

Spark views CPU power in terms of **cores** rather than physical sockets. You control this via configuration properties like `spark.executor.cores` and `spark.executor.instances`.

* **Tasks vs. Cores:** A single executor with 4 cores can execute 4 tasks concurrently. Spark maps partitions of data to tasks. If a DataFrame has 200 partitions and your cluster has 50 total execution cores, Spark processes them in 4 waves ($200 / 50$).
* **Task Granularity (`spark.task.cpus`):** By default, each task consumes 1 CPU core. If you are running heavy machine learning tasks (like Scikit-Learn training inside an `applyInPandas` UDF), you can increase `spark.task.cpus` so that fewer tasks run concurrently per executor, giving each task more CPU breathing room.

---

## **🧠 3. Memory Architecture: The Unified Memory Manager**

Memory management is the single most critical factor in Spark performance. Since Spark 1.6+, Spark uses a **Unified Memory Manager** that allows dynamic boundary shifting between storage and execution memory.

Total executor memory (`spark.executor.memory`) is divided into distinct logical regions:

```
+---------------------------------------------------------------------------------+
|                         Total Executor Memory (JVM Heap)                        |
+---------------------------------+-------------------------------+---------------+
|    Reserved Memory (300 MB)     |       User Memory (~20%)      | Spark Memory  |
| (Hardcoded for internal spark)  | (User data structures, UDFs)  |     (~60%)    |
+---------------------------------+-------------------------------+---------------+

```

### **Breakdown of Memory Pools:**

1. **Reserved Memory:** Fixed size (300 MB by default) reserved strictly for Spark’s internal engine overhead. If your `spark.executor.memory` is set too low (e.g., < 450MB), Spark will throw an error.
2. **User Memory (~20% by default):** Used for holding user-defined data structures, object metadata, internal collections, and data manipulated inside standard Python UDFs. If your Python UDFs or Pandas transformations overflow this region, you get `OutOfMemory (OOM)` errors.
3. **Spark Memory (~60% by default, governed by `spark.memory.fraction`):** The shared pool split dynamically between two sub-regions:
    * **Execution Memory:** Used for shuffles, joins, sorts, and aggregations. If execution needs more space, it can borrow from storage memory (evicting cached blocks if necessary).
    * **Storage Memory:** Used for caching DataFrames (`df.cache()`, `persist()`) and broadcast variables.



---

## **🔬 4. On-Heap vs. Off-Heap Memory & Project Tungsten**

Spark runs on the JVM, which historically suffered from high **Garbage Collection (GC)** overhead and heavy object serialization costs when dealing with gigabytes of data. Spark solves this using **Project Tungsten**:

* **Off-Heap Memory (`spark.memory.offHeap.enabled`):** Spark can allocate memory outside the JVM garbage-collected heap directly through raw memory addresses (using sun.misc.Unsafe). This completely bypasses JVM GC pauses.
* **Binary Row Formats:** Instead of storing data as bulky Java objects (where a small integer has massive JVM header overhead), Tungsten encodes data into compact, custom binary formats. Operations like sorting and hashing are performed directly on these raw binary byte arrays without deserialization.

---

## **🔄 5. Dynamic Allocation**

To prevent static resource starvation (where idle applications lock up cluster nodes), Spark supports **Dynamic Allocation** (`spark.dynamicAllocation.enabled = true`).

* When your workload spikes (e.g., hitting a massive shuffle stage), Spark automatically requests more executors from the cluster manager (YARN/K8s).
* When executors remain idle for a specified timeout window (`spark.dynamicAllocation.executorIdleTimeout`), Spark gracefully decommissions them and releases CPU/RAM back to the cluster.

> **NOTE :**
What about GPUs? As of Spark 3.0, GPU usage has been greatly simplified, but GPUs are still not common stock in Spark instances. GPU, like CPU,
would be in the processing sector of the diagram. For more information, check out the RAPIDS+Spark section on the Nvidia website (https://nvidia.github.io/spark-rapids/Getting-Started/). Most, if not all, cloud providers provide the option to equip your Spark/Databricks cluster with GPU nodes.

![](../images/PySpark_configuration_CPU_RAM.png)

In [7]:
results = (
    spark.read.text("../data/gutenberg_books/*.txt")
    .select(F.split(F.col("value"), " ").alias("line"))
    .select(F.explode(F.col("line")).alias("word"))
    .select(F.lower(F.col("word")).alias("word"))
    .select(F.regexp_extract(F.col("word"), "[a-z']+", 0).alias("word"))
    .where(F.col("word") != "")
    .groupby(F.col("word"))
    .count()
)


results.orderBy(F.col("count").desc()).show(10)

26/11/15 13:42:38 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/gutenberg_books/*.txt.
java.io.FileNotFoundException: File ../data/gutenberg_books/*.txt does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catal

+----+-----+
|word|count|
+----+-----+
| the|39188|
| and|24292|
|  of|21234|
|  to|20581|
|   i|15151|
|   a|14564|
|  in|12857|
|that| 9900|
|  it| 9451|
| was| 8939|
+----+-----+
only showing top 10 rows


In [8]:
results.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (10)
+- HashAggregate (9)
   +- Exchange (8)
      +- HashAggregate (7)
         +- Project (6)
            +- Filter (5)
               +- Generate (4)
                  +- Project (3)
                     +- Filter (2)
                        +- Scan text  (1)


(1) Scan text 
Output [1]: [value#20]
Batched: false
Location: InMemoryFileIndex [file:/workspace/data/gutenberg_books/11-0.txt, ... 5 entries]
ReadSchema: struct<value:string>

(2) Filter
Input [1]: [value#20]
Condition : ((size(split(value#20,  , -1), false) > 0) AND isnotnull(split(value#20,  , -1)))

(3) Project
Output [1]: [split(value#20,  , -1) AS line#21]
Input [1]: [value#20]

(4) Generate
Input [1]: [line#21]
Arguments: explode(line#21), false, [word#24]

(5) Filter
Input [1]: [word#24]
Condition : NOT (regexp_extract(lower(word#24), [a-z']+, 0) = )

(6) Project
Output [1]: [regexp_extract(lower(word#24), [a-z']+, 0) AS word#26]
Input [1]: [word#24]

(7) HashAggregate
Input [1]:

![](../images/Spark_Stage.png)

![](../images/Spark_optimizes_jobs.png)

## **Narrow vs. Wide operations**

In Apache Spark, transformations are categorized into **narrow** or **wide** operations based on how data is distributed across partitions. This distinction is the single most important factor determining whether your job will run lightning-fast in memory or grind to a halt due to network bottlenecks.

---

### **📊 Direct Comparison Matrix**

| Dimension | **Narrow Operations** | **Wide Operations (Shuffles)** |
| --- | --- | --- |
| **Data Movement** | **Local.** Each partition of the parent RDD/DataFrame is consumed by at most one partition of the child. | **Network Shuffle.** Data must be redistributed and shared across multiple nodes in the cluster. |
| **Network I/O** | Zero network traffic between executors. | High network and disk I/O (data is serialized, sent over wire, and spilled to disk). |
| **Stage Boundary** | **No.** Spark can pipeline multiple narrow operations together into a single task stage. | **Yes.** Defines a **Stage Boundary** (a barrier where execution pauses until all shuffle blocks are written and fetched). |
| **Examples** | `filter()`, `select()`, `withColumn()`, `union()`, `map()` | `groupBy()`, `join()`, `distinct()`, `repartition()`, `reduceByKey()` |

---

### **1. Narrow Operations (Pipelined & Local)**

A **narrow dependency** means that each input partition is used by **at most one** output partition.

* **How it Works:** The data needed to compute a child partition lives locally on the same executor (or can be read directly from the source without coordination).
* **The Magic of Pipelining:** Because there is no data shuffling, Spark’s Catalyst Optimizer and Tungsten engine can **pipeline** multiple narrow transformations together. If you chain a `.filter()`, a `.withColumn()`, and a `.select()`, Spark executes them all in a single pass on the row iterator in memory without materializing intermediate results to disk.

```python
# All of these are narrow operations (no shuffle)
clean_df = (
    full_data.where("capacity_GB > 1000")  # Narrow (Filter)
    .withColumn("capacity_TB", F.col("capacity_GB") / 1024)  # Narrow (Transform)
    .select("model", "capacity_TB")  # Narrow (Project)
)

```

---

### **2. Wide Operations (The Shuffle Bottleneck)**

A **wide dependency** means that multiple child partitions depend on data spread across **multiple** parent partitions.

* **Why it Triggers a Shuffle:** To group or join records that share the same key (e.g., grouping all logs by hard drive `model` or joining two tables on `stn`), Spark must collect all matching keys from across *every* partition in the cluster and send them to the correct reducer executor.
* **Stage Boundaries:** Wide operations force Spark to split your execution plan into separate **Stages**.
1. **Stage 1 (Map Phase):** Computes data up to the wide operation and writes shuffle blocks to local disk.
2. **Shuffle Barrier:** Execution stops. Spark waits for all map tasks across all nodes to finish.
3. **Stage 2 (Reduce Phase):** Downstream tasks fetch their respective partition blocks across the network.



```python
# This triggers a WIDE operation (Shuffle) because records with the same model
# scattered across different partitions must be brought together.
summary_df = full_data.groupBy("model").agg(F.count("*"))

```

---

### **⚠️ Performance Engineering Implications**

1. **Minimize Wide Transformations:** Every shuffle is an expensive operation involving disk writing, network serialization, and deserialization. Avoid unnecessary `groupBy` or `join` calls inside iterative loops.
2. **Watch Out for Data Skew:** In wide operations like joins or groupings, if one key accounts for 90% of the rows (e.g., a `NULL` key or a hyper-popular ID), the shuffle will route almost all data to a single executor, causing an **OutOfMemory (OOM)** error or a massive straggler task.
3. **Map-Side Aggregations:** Spark automatically optimizes certain wide operations (like `groupBy().sum()`) by performing a partial local aggregation *before* shuffling, reducing the volume of data sent across the network.

## **Caching a DataFrame**

In Apache Spark, **caching** (or persisting) allows you to store the intermediate results of a DataFrame in memory or disk across cluster nodes. Because Spark’s lineage-based lazy evaluation recomputes transformations from scratch whenever an action is triggered, caching acts as a checkpoint to eliminate redundant computation.

---

### **🔄 `.cache()` vs. `.persist()`**

While often used interchangeably, they have distinct levels of flexibility:

* **`.cache()`**: A shorthand wrapper. It persists the DataFrame using the default storage level: **`MEMORY_AND_DISK`** (deserialized in memory, spilled to disk only if it doesn't fit in RAM).
* **`.persist()`**: Gives you fine-grained control over *where* and *how* data is stored using specific **Storage Levels**.

#### **Common Storage Levels Table**

| Storage Level | Memory? | Disk? | Serialized? | Use Case |
| --- | --- | --- | --- | --- |
| **`MEMORY_ONLY`** (Default for un-serialized) | Yes | No | No | Fast retrieval, but heavy GC overhead if objects are large. |
| **`MEMORY_AND_DISK`** (Spark's `.cache()` default) | Yes | Yes | No | Safest default; spills overflow partitions to disk instead of failing. |
| **`MEMORY_ONLY_SER`** | Yes | No | **Yes** | Serialized as byte arrays; saves massive RAM and GC overhead, but CPU cost to deserialize. |
| **`MEMORY_AND_DISK_SER`** | Yes | Yes | **Yes** | Maximum memory efficiency with disk spill fallback for massive datasets. |

---

### **🧠 How Caching Interacts with Spark Memory**

When you call `.cache()`, data is **not cached immediately**. Caching is a lazy operation; the data is only materialized and stored in the **Storage Memory pool** (managed by Spark's Unified Memory Manager) the *first time* an action is executed on that DataFrame.

1. **Eviction:** If the storage pool fills up, older cached partitions are evicted from memory. Depending on your storage level, evicted partitions are either recomputed from lineage or read from local disk blocks.
2. **Lineage Preservation:** Caching does **not** destroy the DAG lineage. If an executor crashes and loses a cached partition, Spark can still rebuild it by replaying the lineage graph up to that point.

---

### **💻 Code Example: Proper Caching & Unpersisting**

```python
import pyspark.sql.functions as F
from pyspark.storagelevel import StorageLevel

# 1. Build an expensive transformation pipeline (e.g., joins, aggregations)
expensive_df = (
    full_data.groupBy("model", "capacity_GB")
    .agg(
        F.count("*").alias("total_days"),
        F.avg("failure").alias("avg_failure"),
    )
    .where("total_days > 100")
)

# 2. Explicitly persist using a custom storage level (Serialized for memory optimization)
expensive_df.persist(StorageLevel.MEMORY_AND_DISK_SER)

# 3. First Action: Triggers the physical computation AND caches the result in memory/disk
print(f"Total rows passing filter: {expensive_df.count()}")

# 4. Subsequent Actions: Read directly from cache (Blazing fast, zero recomputation)
expensive_df.filter("avg_failure > 0.05").show(5)
expensive_df.select(F.sum("total_days")).show()

# 5. Best Practice: Free up cluster memory when the DataFrame is no longer needed
expensive_df.unpersist()
```

---

### **⚠️ When to Cache (and When NOT to)**

* **DO Cache When:**
    * A DataFrame is accessed **multiple times** in your workflow (e.g., iterative machine learning training loops, repeated joins against the same lookup table, or generating multiple reports from a single cleansed base table).
    * The upstream transformation graph is computationally expensive (heavy shuffles, complex multi-stage joins).


* **DO NOT Cache When:**
    * The DataFrame is only used **once** (caching adds overhead for writing blocks to memory/disk and tracking block locations in the Driver).
    * The dataset is larger than available cluster RAM and lacks a disk-spill storage level, or causes constant memory thrashing that evicts other critical execution blocks.

# **Summary**

This excerpt captures the core architectural principles that govern Apache Spark's high-performance distributed computing engine. Below is a detailed, technical breakdown of each concept, explaining how Spark manages memory, executes workloads, and optimizes data pipelines.

---

## **1. Dual Memory Architecture: Storage vs. Execution Memory**

Spark relies heavily on RAM to achieve its speed, avoiding the heavy disk I/O penalties of traditional MapReduce. Spark's **Unified Memory Manager** dynamically divides executor memory into two primary pools:

* **Storage Memory:** Dedicated to caching DataFrames (`.cache()` or `.persist()`) and storing broadcast variables. When you cache data, Spark loads it into this pool so subsequent actions can read it directly from memory without recomputing the lineage.
* **Execution Memory (Operational):** Allocated for temporary data structures required during computational operations, such as shuffles, joins, sorts, and aggregations.

**The Dynamic Boundary:** These two pools share a flexible boundary. If Storage Memory is not being used, Execution Memory can borrow its space, and vice versa. If memory pressure exceeds limits, Spark automatically spills excess data to local disk and brings it back when needed. Memory allocation is configured when instantiating the `SparkSession` (e.g., adjusting `spark.executor.memory` or `spark.memory.fraction`).

---

## **2. Cluster Observability via the Spark UI**

The **Spark Web UI** (running by default on port `4040` during an active session) is an essential diagnostic tool for monitoring execution health and debugging performance bottlenecks. Key tabs include:

* **Environment Tab:** Displays detailed configurations, including JVM flags, classpaths, OS details, CPU core counts, and system libraries loaded across executors.
* **Executors Tab:** Shows active resource distribution, memory consumption per executor, garbage collection (GC) time, and active task loads.
* **Jobs / Stages / SQL Tabs:** Track execution progress, shuffling metrics, and physical query plans in real time.

---

## **3. The Execution Lifecycle: Jobs, Transformations, and Actions**

Spark operates under **lazy evaluation**, meaning it does not execute code the moment you write it.

* **Transformations** (e.g., `.select()`, `.filter()`, `.groupBy()`) are lazy. They simply append nodes to a logical Directed Acyclic Graph (DAG) representing the data lineage.
* **Actions** (e.g., `.count()`, `.show()`, `.write()`, `.collect()`) are eager. Calling an action triggers Spark to turn the DAG into an execution plan, packages it into a **Job**, and dispatches it to the cluster. A single Spark application can contain multiple sequential jobs if multiple actions are invoked.

---

## **4. Stages, Exchanges, and Physical Plan Inspection**

When a job is triggered, the Catalyst Optimizer compiles the logical plan into a physical execution plan, which is broken down hierarchically: **Job $\rightarrow$ Stages $\rightarrow$ Tasks**.

* **Stages:** Logical units of work separated by **Exchange (Shuffle) operations**. Whenever data must move across the network between worker nodes, Spark closes the current stage and opens a new one.
* **Inspection:** You can inspect how Spark plans to execute your code using the `.explain()` method (e.g., `df.explain(True)`), which outputs the parsed, analyzed, optimized, and physical plans. Alternatively, the **SQL Tab** in the Spark UI provides a visual DAG of these stages.

---

## **5. Narrow vs. Wide Operations**

The distinction between these two operation types determines whether a workflow runs efficiently in memory or triggers a costly network bottleneck:

* **Narrow Operations:** Each partition of the parent DataFrame is consumed by at most one partition of the child DataFrame (e.g., `.filter()`, `.select()`, `.withColumn()`). Spark can **pipeline** these operations together locally on an executor without moving data across the network.
* **Wide Operations:** Require data from multiple parent partitions to be combined, triggering a **shuffle or exchange** across cluster nodes (e.g., `.groupBy()`, `.join()`, `.distinct()`). Data is serialized, written to local executor disks, sent across the network, and re-read by reducer tasks, forming a hard stage boundary.

---

## **6. Caching Mechanics & Best Practices**

Caching moves a DataFrame's partitions from transient execution lineage into **Storage Memory** (with automatic spilling to disk if memory is exhausted).

* **The Optimization Trade-off:** While caching speeds up repetitive access, it can interfere with the **Catalyst Optimizer**. If you cache a DataFrame too early, Spark locks in that physical representation, preventing the optimizer from pushing down subsequent filters or pruning unused columns through upstream transformations.
* **When to Cache:** Caching is counterproductive in standard linear ETL pipelines where a DataFrame is read, transformed, and written once. It is ideal for **iterative algorithms or machine learning training loops**, where the exact same dataset must be scanned dozens or hundreds of times across multiple epochs.